In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Crée les dossiers si nécessaire
os.makedirs("data/raw", exist_ok=True)

# Télécharger population si pas déjà présent
if not os.path.exists("data/raw/population_communes.xlsx"):
    from src.fetch_population import download_population
    download_population()

# Télécharger élections 1er tour si pas déjà présent
if not os.path.exists("data/raw/elections_t1_legislatives.xlsx"):
    from src.fetch_elections_t1 import download_elections
    download_elections()

# Charger population
df_pop = pd.read_excel("data/raw/population_communes.xlsx")

# Charger élections 1er tour
df_ele1 = pd.read_excel("data/raw/elections_t1_legislatives.xlsx")

# Vérifier les colonnes
print("Population :", df_pop.columns.tolist())
print("Élections 1er tour :", df_ele1.columns.tolist())


Population : ['objectid', 'reg', 'dep', 'cv', 'codgeo', 'libgeo', 'p13_pop', 'p14_pop', 'p15_pop', 'p16_pop', 'p17_pop', 'p18_pop', 'p19_pop', 'p20_pop', 'p21_pop']
Élections 1er tour : ['Code du département', 'Libellé du département', 'Code de la circonscription', 'Libellé de la circonscription', 'Code de la commune', 'Libellé de la commune', 'Etat saisie', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Nuance', 'Voix', '% Voix/Ins', '% Voix/Exp', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 

In [11]:
df_pop_clean = df_pop[['codgeo', 'p13_pop']].copy()
df_pop_clean.columns = ['code_commune', 'population']
df_pop_clean['code_commune'] = df_pop_clean['code_commune'].astype(str).str.zfill(5)
df_pop_clean['population'] = df_pop_clean['population'].astype(int)


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [3]:
# Colonnes importantes : code commune et population (p13_pop)
df_pop_clean = df_pop[['codgeo', 'p13_pop']].copy()
df_pop_clean.columns = ['code_commune', 'population']
df_pop_clean['code_commune'] = df_pop_clean['code_commune'].astype(str).str.zfill(5)


In [4]:
colonnes_utiles2 = [ 
    'Code de la commune', 'Libellé de la commune',
    'Code du département', 'Libellé du département',
    'Code de la circonscription', 'Libellé de la circonscription',
    'Exprimés'
]

df_ele2_clean = df_ele2[colonnes_utiles2].copy()
df_ele2_clean.columns = [
    'code_commune', 'nom_commune',
    'code_departement', 'nom_departement',
    'code_circonscription', 'nom_circonscription',
    'voix'
]
df_ele2_clean['code_commune'] = df_ele2_clean['code_commune'].astype(str).str.zfill(5)

# Merge population avec 2nd tour
df_merged2 = df_ele2_clean.merge(df_pop_clean, on='code_commune', how='left')


In [5]:
# Colonnes fixes pour localisation
loc_cols = ['Code du département','Code de la commune','Code de la circonscription']

# Colonnes des partis et voix : colonnes Z/AA, AH/AI, AP/AQ, ...
# On les transforme en long format automatiquement
parti_cols = [
    ('Z','AA'), ('AH','AI'), ('AP','AQ'), ('AX','AY'), ('BF','BG'),
    ('BN','BO'), ('BV','BW'), ('CD','CE'), ('CL','CM'), ('CT','CU'),
    ('DB','DC'), ('DJ','DK'), ('DR','DS'), ('DZ','EA'), ('EH','EI'),
    ('EP','EQ'), ('EX','EY'), ('FF','FG'), ('FN','FO'), ('FV','FW'),
    ('GD','GE'), ('GL','GM')
]

rows = []
for idx, row in df_ele1.iterrows():
    code_dept = str(row['Code du département']).zfill(2)
    code_com = str(row['Code de la commune']).zfill(3)
    code_circ = str(row['Code de la circonscription']).zfill(2)
    for col_parti, col_voix in parti_cols:
        parti = row.get(col_parti)
        voix = row.get(col_voix)
        if pd.notna(parti) and pd.notna(voix) and voix > 0:
            rows.append({
                'code_departement': code_dept,
                'code_commune': code_com,
                'code_circonscription': code_circ,
                'parti': parti,
                'voix': voix
            })

df_ele1_long = pd.DataFrame(rows)


In [8]:
# Créer code_commune compatible avec df_pop_clean
df_ele1_long['code_commune'] = df_ele1_long['code_departement'].astype(str).str.zfill(2) + \
                                df_ele1_long['code_commune'].astype(str).str.zfill(3)

# Merge avec population
df_ele1_long = df_ele1_long.merge(df_pop_clean, on='code_commune', how='left')

# Supprimer les lignes où la population est NaN
df_ele1_long = df_ele1_long.dropna(subset=['population'])
df_ele1_long['population'] = df_ele1_long['population'].astype(int)

# Vérification
print(df_ele1_long.head())
print("Nombre de lignes après merge :", len(df_ele1_long))


KeyError: 'code_departement'

In [9]:
print(df_pop.columns)
print(df_ele1.columns)


Index(['objectid', 'reg', 'dep', 'cv', 'codgeo', 'libgeo', 'p13_pop',
       'p14_pop', 'p15_pop', 'p16_pop', 'p17_pop', 'p18_pop', 'p19_pop',
       'p20_pop', 'p21_pop'],
      dtype='object')
Index(['Code du département', 'Libellé du département',
       'Code de la circonscription', 'Libellé de la circonscription',
       'Code de la commune', 'Libellé de la commune', 'Etat saisie',
       'Inscrits', 'Abstentions', '% Abs/Ins',
       ...
       'Unnamed: 187', 'Unnamed: 188', 'Unnamed: 189', 'Unnamed: 190',
       'Unnamed: 191', 'Unnamed: 192', 'Unnamed: 193', 'Unnamed: 194',
       'Unnamed: 195', 'Unnamed: 196'],
      dtype='object', length=197)
